# 04 — Forecasting Layer 2: Uncertainty & Confidence

Builds on Layer 1 models to add:
1. Prophet 70% confidence intervals (native)
2. XGBoost quantile regression (lower / upper bounds)
3. Historical accuracy by context (day-of-week × event flag)
4. Reliability flag: HIGH / MEDIUM / LOW

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import pickle
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from src.utils.db_connect import get_engine
from src.features.feature_engineering import build_features

engine = get_engine()
MODEL_DIR = Path('../src/models')
print('Connected')

## 1. Load Data & Re-apply Features

In [ ]:
df = pd.read_sql("""
    SELECT dd.*, ds.day_of_week, ds.event_flag, ds.temperature_index,
           ds.is_public_holiday, ds.trading_hours, ds.promo_flag
    FROM division_daily dd
    JOIN daily_sales ds ON dd.date = ds.date
    ORDER BY dd.date, dd.division_code
""", engine, parse_dates=['date'])

df = build_features(df)

split_date = df['date'].max() - pd.Timedelta(weeks=8)
train = df[df['date'] <= split_date].copy()
test  = df[df['date'] >  split_date].copy()
print(f'Test period: {test["date"].min().date()} → {test["date"].max().date()}')

## 2. Prophet Confidence Intervals (70% CI)

In [ ]:
DEMO_DIV = 34  # Men's Cut & Sewn

with open(MODEL_DIR / f'prophet_div{DEMO_DIV}.pkl', 'rb') as f:
    m_prophet = pickle.load(f)

d_test = test[test['division_code'] == DEMO_DIV][['date', 'sales_amt', 'event_flag']].copy()
d_test = d_test.rename(columns={'date': 'ds', 'sales_amt': 'y'})
for ev in ['Arigato', 'Christmas', 'EOFY', 'Vivid', 'LongWeekend']:
    d_test[ev] = (d_test['event_flag'] == ev).astype(int)

future_df = d_test[['ds'] + ['Arigato','Christmas','EOFY','Vivid','LongWeekend']]
forecast = m_prophet.predict(future_df)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat_upper'],
    fill=None, mode='lines', line_color='rgba(33,150,243,0)', name='Upper 70% CI'
))
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat_lower'],
    fill='tonexty', mode='lines', line_color='rgba(33,150,243,0)',
    fillcolor='rgba(33,150,243,0.15)', name='70% CI Band'
))
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat'],
    mode='lines', name='Prophet Forecast', line=dict(color='#2196F3')
))
fig.add_trace(go.Scatter(
    x=d_test['ds'], y=d_test['y'],
    mode='lines', name='Actual', line=dict(color='#F44336')
))
fig.update_layout(
    title=f'Prophet — 70% Confidence Interval (Div {DEMO_DIV}: Men\'s Cut & Sewn)',
    yaxis_tickformat='$,.0f', hovermode='x unified'
)
fig.show()

## 3. XGBoost Quantile Regression (Lower + Upper Bounds)

In [ ]:
import xgboost as xgb

FEATURE_COLS = [
    'division_code', 'temperature_index', 'trading_hours',
    'is_public_holiday', 'promo_flag', 'dow_num', 'month', 'day_of_year',
    'event_Arigato', 'event_Christmas', 'event_EOFY', 'event_Vivid', 'event_LongWeekend',
    'lag_7', 'lag_14', 'lag_28', 'rolling_7_mean', 'rolling_14_mean',
]

X_train = train[FEATURE_COLS].fillna(0)
y_train = train['sales_amt']
X_test  = test[FEATURE_COLS].fillna(0)
y_test  = test['sales_amt']

# Lower bound: alpha=0.15 (15th percentile)
xgb_lower = xgb.XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    objective='reg:quantileerror', quantile_alpha=0.15,
    random_state=42
)
xgb_lower.fit(X_train, y_train)

# Upper bound: alpha=0.85 (85th percentile)
xgb_upper = xgb.XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    objective='reg:quantileerror', quantile_alpha=0.85,
    random_state=42
)
xgb_upper.fit(X_train, y_train)

pred_lower = xgb_lower.predict(X_test)
pred_upper = xgb_upper.predict(X_test)

# Coverage: fraction of actuals within [lower, upper]
coverage = np.mean((y_test.values >= pred_lower) & (y_test.values <= pred_upper))
print(f'XGBoost quantile 70% CI coverage: {coverage:.1%} (target: ~70%)')

# Save quantile models
with open(MODEL_DIR / 'xgb_quantile_lower.pkl', 'wb') as f:
    pickle.dump(xgb_lower, f)
with open(MODEL_DIR / 'xgb_quantile_upper.pkl', 'wb') as f:
    pickle.dump(xgb_upper, f)
print('Quantile models saved.')

## 4. Historical Accuracy by Context (day-of-week × event)

In [ ]:
# Use the median XGBoost point model from Layer 1
with open(MODEL_DIR / 'xgb_layer1.pkl', 'rb') as f:
    xgb_median = pickle.load(f)

train_pred = xgb_median.predict(train[FEATURE_COLS].fillna(0))
train_errors = train.copy()
train_errors['predicted'] = train_pred
train_errors['abs_pct_error'] = np.abs(
    (train_errors['sales_amt'] - train_errors['predicted']) / train_errors['sales_amt']
) * 100

accuracy_lookup = (
    train_errors.groupby(['day_of_week', 'event_flag'])['abs_pct_error']
    .mean()
    .reset_index()
    .rename(columns={'abs_pct_error': 'mean_mape'})
)

# Save lookup for dashboard
accuracy_lookup.to_csv(MODEL_DIR / 'accuracy_lookup.csv', index=False)
print('Accuracy lookup table:')
print(accuracy_lookup.sort_values('mean_mape', ascending=False).head(10))

## 5. Reliability Flag Logic

In [ ]:
def get_reliability_flag(dow: str, event: str, accuracy_lookup: pd.DataFrame) -> tuple[str, str]:
    """
    Returns (flag, explanation) where flag is HIGH / MEDIUM / LOW.
    HIGH  < 5% historical MAPE for this context
    MEDIUM 5–12%
    LOW   >12% OR event week (inherently uncertain)
    """
    if event in ('Arigato', 'Christmas', 'Vivid'):
        return 'LOW', f'Event week ({event}) — historical forecast accuracy is lower for irregular events.'

    match = accuracy_lookup[
        (accuracy_lookup['day_of_week'] == dow) &
        (accuracy_lookup['event_flag'] == event)
    ]
    if len(match) == 0:
        return 'MEDIUM', 'Insufficient history for this context.'

    mape = match.iloc[0]['mean_mape']
    if mape < 5:
        return 'HIGH', f'Historical MAPE for {dow} / {event}: {mape:.1f}% — reliable context.'
    elif mape < 12:
        return 'MEDIUM', f'Historical MAPE for {dow} / {event}: {mape:.1f}%.'
    else:
        return 'LOW', f'Historical MAPE for {dow} / {event}: {mape:.1f}% — forecasts are less reliable in this context.'

# Demo
flag, expl = get_reliability_flag('Thursday', 'None', accuracy_lookup)
print(f'Flag: {flag}\nExplanation: {expl}')

## 6. Uncertainty Summary Chart

In [ ]:
# Aggregate test predictions to daily store level
with open(MODEL_DIR / 'xgb_layer1.pkl', 'rb') as f:
    xgb_median = pickle.load(f)

test_copy = test.copy()
test_copy['pred_median'] = xgb_median.predict(X_test)
test_copy['pred_lower']  = pred_lower
test_copy['pred_upper']  = pred_upper

daily_agg = test_copy.groupby('date').agg(
    actual=('sales_amt', 'sum'),
    pred_median=('pred_median', 'sum'),
    pred_lower=('pred_lower', 'sum'),
    pred_upper=('pred_upper', 'sum'),
).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_agg['date'], y=daily_agg['pred_upper'],
    fill=None, mode='lines', line_color='rgba(33,150,243,0)', showlegend=False
))
fig.add_trace(go.Scatter(
    x=daily_agg['date'], y=daily_agg['pred_lower'],
    fill='tonexty', mode='lines', line_color='rgba(33,150,243,0)',
    fillcolor='rgba(33,150,243,0.2)', name='70% CI'
))
fig.add_trace(go.Scatter(
    x=daily_agg['date'], y=daily_agg['pred_median'],
    mode='lines', name='Forecast', line=dict(color='#2196F3', dash='dash')
))
fig.add_trace(go.Scatter(
    x=daily_agg['date'], y=daily_agg['actual'],
    mode='lines', name='Actual', line=dict(color='#F44336')
))
fig.update_layout(
    title='Forecast with 70% Confidence Interval — Test Period (All Divisions)',
    yaxis_tickformat='$,.0f', hovermode='x unified'
)
fig.show()